# Week 5 Data QC — v06 RC1 (12/10/2026)
Companion evidence for the Data Lead PR. Run all cells from the repository or notebook directory.
Inputs: `clauses_v06.csv`, metadata, committed raw text, taxonomy v0.3, guideline v1.0, and the existing provisional split.
The candidate inherits clause labels from the reviewed snapshot. No legal risk labels are created. Independent adjudication is pending.


In [1]:
from pathlib import Path
import csv, json, hashlib, itertools, re
from collections import Counter, defaultdict
from dataclasses import asdict
import sys
ROOT = Path.cwd()
if not (ROOT / 'data').is_dir(): ROOT = ROOT.parent
assert (ROOT / 'scripts/validate_dataset.py').is_file()
sys.path.insert(0, str(ROOT))
from scripts.validate_dataset import validate_dataset
from scripts.build_offsets import normalize_whitespace
from scripts.build_provisional_split import load_current_data, build_manifest, normalized, overlap_reason
def sha(path): return hashlib.sha256((ROOT / path).read_bytes()).hexdigest()
def write_json(path, obj):
    p = ROOT / path; p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(json.dumps(obj, ensure_ascii=False, indent=2) + '\n', encoding='utf-8', newline='\n')
def write_csv(path, records):
    p = ROOT / path; p.parent.mkdir(parents=True, exist_ok=True)
    with p.open('w', encoding='utf-8', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=list(records[0]), lineterminator='\n'); writer.writeheader(); writer.writerows(records)
rows, metadata = load_current_data(ROOT, 'v06')
byid = {r['clause_id']: r for r in rows}
report = validate_dataset(ROOT / 'data/processed/clauses_v06.csv', ROOT / 'data/contracts_metadata.csv', ROOT / 'data/processed/raw_text')
assert not report.errors, report.errors
raw = {c: (ROOT / f'data/processed/raw_text/{c}.txt').read_text(encoding='utf-8') for c in metadata}
print('Validator:', len(report.errors), 'errors;', report.clause_count, 'clauses;', report.offset_quality_counts)


Validator: 0 errors; 234 clauses; {'CONTEXT': 9, 'EXACT': 225}


## Duplicate and annotation consistency
Whitespace normalization reproduces the validator. Casefold normalization reproduces the split filter. Near similarity uses the existing 0.85 threshold, minimum 30 characters and 0.85 length ratio. Different labels on near text create review candidates, not automatic relabels; placeholder runs can inflate this character metric.


In [2]:
ws, folded = defaultdict(list), defaultdict(list)
for r in rows:
    ws[normalize_whitespace(r['clause_text'])].append(r)
    folded[normalized(r['clause_text'])].append(r)
dup = [v for v in folded.values() if len(v) > 1]
conflicts = [v for v in dup if len({r['clause_type'] for r in v}) > 1]
duplicates = [{'clause_ids': '|'.join(r['clause_id'] for r in v), 'count': len(v),
               'labels': '|'.join(sorted({r['clause_type'] for r in v})),
               'similarity_groups': '|'.join(sorted({metadata[r['contract_id']]['similarity_group'] for r in v})),
               'normalized_text': normalized(v[0]['clause_text'])} for v in dup]
write_csv('reports/week5/duplicate_groups.csv', duplicates)
near_pairs = []
for a,b in itertools.combinations(rows, 2):
    if a['clause_type'] == b['clause_type']: continue
    match = overlap_reason(normalized(a['clause_text']), normalized(b['clause_text']), .85)
    if match:
        near_pairs.append({'left_id': a['clause_id'], 'right_id': b['clause_id'], 'similarity': match[1],
                           'left_label': a['clause_type'], 'right_label': b['clause_type'],
                           'left_text': a['clause_text'], 'right_text': b['clause_text'],
                           'review_status': 'PENDING_TAXONOMY_REVIEW',
                           'review_hint': 'Check semantic topic and actor; discount unfilled template placeholders.'})
write_csv('reports/week5/annotation_review_pairs.csv', near_pairs)
assert not conflicts
print('Whitespace duplicate groups/affected/extra:', sum(len(v)>1 for v in ws.values()), sum(len(v) for v in ws.values() if len(v)>1), report.duplicate_text_count)
print('Casefold duplicate groups/affected/extra:', len(dup), sum(map(len,dup)), sum(len(v)-1 for v in dup))
print('Exact normalized label conflicts:', len(conflicts), '; near pairs with different labels:', len(near_pairs))


Whitespace duplicate groups/affected/extra: 31 64 33
Casefold duplicate groups/affected/extra: 31 65 34
Exact normalized label conflicts: 0 ; near pairs with different labels: 23


## Split reproducibility and leakage
Whole template groups remain in one split. Evaluation exclusions are masks: retain the full dataset and remove the listed IDs only from scoring. The check establishes absence of matches under the declared text rule, not absence of semantic or prior model exposure leakage.


In [3]:
split_path = 'data/splits/provisional_v06.json'
saved = json.loads((ROOT / split_path).read_text(encoding='utf-8'))
rebuilt = build_manifest(rows, metadata, 'v06'); rebuilt['dataset_sha256'] = sha('data/processed/clauses_v06.csv')
assert saved == rebuilt
active = {s: [r for r in rows if r['contract_id'] in v['contracts'] and r['clause_id'] not in v['excluded_clause_ids']] for s,v in saved['splits'].items()}
groups = [set(v['similarity_groups']) for v in saved['splits'].values()]
assert all(not a & b for a,b in itertools.combinations(groups,2))
remaining = []
for a,b in itertools.combinations(active,2):
    for x,y in itertools.product(active[a],active[b]):
        if overlap_reason(normalized(x['clause_text']), normalized(y['clause_text']), .85):
            remaining.append((x['clause_id'],y['clause_id']))
assert not remaining
for s,v in saved['splits'].items():
    print(s, 'total/scorable/exact:', v['clause_count'], v['scorable_clause_count'], v['exact_span_scorable_count'])
print('Exclusion reasons:', dict(Counter(e['reason'] for e in saved['exclusions'])))
print('Remaining active cross-split pairs:', len(remaining))


train total/scorable/exact: 112 112 110
dev total/scorable/exact: 79 49 48
test total/scorable/exact: 43 34 28
Exclusion reasons: {'EXACT_CROSS_SPLIT': 6, 'NEAR_CROSS_SPLIT': 33}
Remaining active cross-split pairs: 0


## Hard test candidate and source evidence
Select contrasting actors/topics, multi-topic CONTEXT cases, and unfilled template cases from scorable test `SIM_C` only. All cases carry inherited expected clause labels, source hashes, literal excerpts and taxonomy references. CONTEXT rows have no gold exact clause span. Classification and benchmark use require independent reviewer sign-off; legal risk ground truth is unavailable.


In [4]:
selection = {
  '003': ('trial_period_dates', 'taxonomy v0.3 section 2 CONTRACT_TERM; reviewer must confirm probation treatment'),
  '005': ('open_job_scope', 'taxonomy v0.3 section 2 JOB_INFO'),
  '010': ('insurance_payment_vs_wage', 'taxonomy v0.3 sections 2 and 4.2 INSURANCE_SAFETY'),
  '020': ('ppe_vs_working_conditions', 'taxonomy v0.3 section 4.2'),
  '021': ('housing_vs_ppe', 'taxonomy v0.3 section 4.2'),
  '022': ('employee_entitlement_vs_employer_payment_duty', 'taxonomy v0.3 section 4.3; review actor from source section'),
  '023': ('split_leave_context', 'guideline v1.0 S4/S5; taxonomy v0.3 section 4.1'),
  '024': ('split_training_context', 'guideline v1.0 S4/S5; taxonomy v0.3 section 2 TRAINING'),
  '025': ('split_housing_context', 'guideline v1.0 S4/S5; taxonomy v0.3 section 4.2'),
  '026': ('split_ppe_context', 'guideline v1.0 S4/S5; taxonomy v0.3 section 4.2'),
  '027': ('employee_damage_right_taxonomy_gap', 'taxonomy v0.3 section 4.6 OTHER; guideline v1.0 section 7.5'),
  '029': ('employee_safety_compliance_vs_safety_topic', 'taxonomy v0.3 section 4.4'),
  '030': ('employee_compensation_liability', 'taxonomy v0.3 section 4.6'),
  '033': ('employer_compensation_right', 'taxonomy v0.3 section 4.6'),
  '034': ('employer_payment_duty_vs_benefit', 'taxonomy v0.3 section 4.3'),
  '035': ('housing_topic_with_employer_actor', 'taxonomy v0.3 section 4.2'),
  '038': ('reprimand_vs_termination_unfilled', 'taxonomy v0.3 section 2 EMPLOYEE_OBLIGATIONS_DISCIPLINE'),
  '039': ('dismissal_vs_reprimand_unfilled', 'taxonomy v0.3 section 4.5'),
  '041': ('employer_compensation_liability', 'taxonomy v0.3 section 4.6'),
  '043': ('effective_date_repeated_subject_context', 'guideline v1.0 S4/S5; taxonomy v0.3 section 2 CONTRACT_TERM'),
}
hard = []
test_ids = {r['clause_id'] for r in active['test']}
for suffix,(tag,basis) in selection.items():
    r = byid[f'HDLD004_C{suffix}']; assert r['clause_id'] in test_ids
    text = raw[r['contract_id']]; start,end = int(r['start_offset']),int(r['end_offset'])
    excerpt = text[start:end]
    lo = text.rfind('\n',0,start) + 1; hi = text.find('\n',end)
    if hi < 0: hi = len(text)
    exact = r['offset_quality'] == 'EXACT'
    if exact: assert normalize_whitespace(excerpt) == normalize_whitespace(r['clause_text'])
    hard.append({'case_id': 'W5_' + r['clause_id'], 'clause_id': r['clause_id'], 'contract_id': r['contract_id'],
                 'similarity_group': 'SIM_C', 'origin_split': 'test', 'input_text': r['clause_text'],
                 'section_title': r['section_title'], 'expected_clause_type': r['clause_type'],
                 'annotation_status_in_source': r['annotation_status'], 'expected_offset_quality': r['offset_quality'],
                 'source_reference_span': {'start': start, 'end': end, 'text': excerpt},
                 'gold_exact_clause_span': {'start': start, 'end': end} if exact else None,
                 'source_parent_line': text[lo:hi], 'source_raw_text': f'data/processed/raw_text/{r["contract_id"]}.txt',
                 'source_raw_sha256': sha(f'data/processed/raw_text/{r["contract_id"]}.txt'),
                 'source_dataset_sha256': sha('data/processed/clauses_v06.csv'), 'source_notes': r['notes'],
                 'hard_case_tag': tag, 'taxonomy_basis': basis, 'independent_review_status': 'PENDING',
                 'legal_risk_ground_truth': None,
                 'scoring_policy': 'Exact span allowed only for EXACT; classification after independent adjudication.'})
p = ROOT / 'data/evaluation/hard_test_v06_rc1.jsonl'; p.parent.mkdir(parents=True,exist_ok=True)
p.write_text(''.join(json.dumps(x,ensure_ascii=False) + '\n' for x in hard),encoding='utf-8',newline='\n')
assert len(hard) == 20 and len({x['case_id'] for x in hard}) == 20
print('Hard candidate:', len(hard), 'cases;', len({x['expected_clause_type'] for x in hard}), 'labels;', dict(Counter(x['expected_offset_quality'] for x in hard)))


Hard candidate: 20 cases; 11 labels; {'EXACT': 14, 'CONTEXT': 6}


## Report and release manifest
RC1 is ready for review. The split and hard test are provisional. Original document hashes are checked against the extraction receipt. Reports preserve the initial failure and its remediation in `docs/WEEK5_DATA_QC_REPORT.md`.


In [5]:
receipt = (ROOT / 'data/processed/raw_text/README.md').read_text(encoding='utf-8')
source_hashes = {}
for filename,digest in re.findall(r'\| (HDLD\d+\.docx?) \| `([0-9a-f]{64})`', receipt):
    if filename not in {item['file_name'] for item in metadata.values()}: continue
    path = f'data/raw/contracts/{filename}'; assert sha(path) == digest, filename; source_hashes[path] = digest
assert len(source_hashes) == 6
warnings = [
 {'code':'CONTEXT_NOT_EXACT','count':9,'impact':'Exclude CONTEXT from exact clause-span metrics.'},
 {'code':'DUPLICATE_TEXT','count':33,'impact':'Extra rows after whitespace normalization; retain and mask for evaluation.'},
 {'code':'EVALUATION_EXCLUSIONS','count':39,'impact':'30 dev + 9 test rows excluded from scoring.'},
 {'code':'NEAR_TEXT_DIFFERENT_LABELS','count':len(near_pairs),'impact':'Review semantic topic/actor and placeholders; no automatic relabel.'},
 {'code':'LOW_CLASS_SUPPORT','label':'WORKING_CONDITIONS','count':7,'impact':'Limited class support; collect independent template families.'},
 {'code':'TEST_CLASS_MISSING','label':'WORKING_TIME','count':0,'impact':'Scorable test cannot validate all 12 labels.'},
 {'code':'DEV_CLASS_MISSING','label':'TERMINATION','count':0,'impact':'No dev support for this class.'},
 {'code':'INDEPENDENT_GOLD_REVIEW_PENDING','count':20,'impact':'Hard candidate cannot be a locked benchmark before adjudication.'},
 {'code':'SOURCE_URLS_UNRESOLVED','contracts':['HDLD001','HDLD002','HDLD003'],'impact':'Original URLs remain TODO in data_sources.md.'},
 {'code':'HDLD007_INTAKE_BLOCKED','impact':'Raw not committed pending redistribution permission; absent from v06 release metadata and dataset.'},
 {'code':'SEMANTIC_LEAKAGE_UNASSESSED','impact':'Character filtering does not prove no semantic or historical model exposure leakage.'},
 {'code':'LEGAL_RISK_GOLD_UNAVAILABLE','impact':'Reviewed clause types do not verify legal risk, citation validity, or current law.'},
]
audit = {'audit_date':'2026-10-12','dataset':'data/processed/clauses_v06.csv','dataset_sha256':sha('data/processed/clauses_v06.csv'),
         'validator':asdict(report),'contract_counts':dict(Counter(r['contract_id'] for r in rows)),
         'whitespace_duplicates':{'groups':sum(len(v)>1 for v in ws.values()),'affected_rows':sum(len(v) for v in ws.values() if len(v)>1),'extra_rows':report.duplicate_text_count},
         'casefold_duplicates':{'groups':len(dup),'affected_rows':sum(map(len,dup)),'extra_rows':sum(len(v)-1 for v in dup)},
         'exact_label_conflicts':len(conflicts),'near_different_label_pairs':len(near_pairs),
         'split_reproducible':saved==rebuilt,'active_cross_split_matches':len(remaining),'split_summary':saved['splits'],
         'warnings':warnings,'raw_document_hashes_verified':source_hashes,
         'hard_test':{'cases':len(hard),'labels':len({x['expected_clause_type'] for x in hard}),'qualities':dict(Counter(x['expected_offset_quality'] for x in hard))}}
write_json('reports/week5/qc_summary.json',audit)
included = ['data/processed/clauses_v06.csv','data/contracts_metadata.csv',split_path,'data/evaluation/hard_test_v06_rc1.jsonl',
            'docs/annotation_guideline.md','docs/clause_taxonomy_v03.md','docs/dataset_schema.md','docs/data_sources.md',
            'docs/WEEK5_DATA_QC_REPORT.md','docs/WEEK5_AI_EVALUATION_HANDOFF.md','docs/CONTRACT_INTAKE_QC_2026-10-05.md',
            'data/releases/v06-rc1/README.md','data/processed/raw_text/README.md',
            'scripts/validate_dataset.py','scripts/build_provisional_split.py',
            'reports/week5/qc_summary.json','reports/week5/duplicate_groups.csv','reports/week5/annotation_review_pairs.csv']
included += [f'data/processed/raw_text/HDLD{i:03d}.txt' for i in range(1,7)] + list(source_hashes)
manifest = {'release_id':'v06-rc1','created_on':'2026-10-12','status':'CANDIDATE_REVIEW_REQUIRED',
            'source_data_commit':'5ce5796','base_main_commit':'3fadcfe','dataset_version':'v06',
            'scope':'Clause taxonomy and source spans; no legal risk gold labels',
            'machine_qc':'PASS','benchmark_status':'PROVISIONAL_NOT_LOCKED','files':{p:sha(p) for p in included},
            'required_signoffs':['Data reviewer: provenance, near-label pairs and source context','AI/Evaluation Lead: gold adjudication, masks and metric denominators'],
            'blockers':['Independent hard-test adjudication pending','Test missing WORKING_TIME and dev missing TERMINATION','Semantic leakage review pending','Legal risk ground truth unavailable'],
            'pending_intake':['HDLD007: redistribution permission and annotation pending']}
write_json('data/releases/v06-rc1/manifest.json',manifest)
print('RC:',manifest['status'],'; verified file hashes:',len(manifest['files']))
print('Warnings:',len(warnings),'categories; validator errors:',len(report.errors))


RC: CANDIDATE_REVIEW_REQUIRED ; verified file hashes: 30
Warnings: 12 categories; validator errors: 0


## Takeaways
234 reviewed clause records / 5 contracts / 4 template groups; 225 EXACT and 9 CONTEXT. Structural validator passes. Whitespace duplicate extras: 33; casefold duplicate extras: 34. No identical normalized text has conflicting labels. There are 23 near-text pairs with differing labels awaiting taxonomy review. Scoring masks leave train/dev/test 112/49/34 rows, including 110/48/28 exact spans. Hard candidate: 20 test cases, 14 EXACT and 6 CONTEXT across 11 labels. Independent gold review and final benchmark approval remain open.
